# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane check: I am keeping Lane 2, Refresh / Content Opportunity Scoring.** Lanes lock this week and my question is the same as in W01: which pages should a reviewer look at first? This notebook builds the simple rule that my Week-5 model has to beat.

It uses the same page frame as W03 (`w03_data_contract.ipynb`): **one row = one page**, features from March 2026 only and the label `declined_next_month` from April 2026. The notebook does three things:

1. checks two signals my rule leans on, each with a bucket table, n and a one word verdict (section 1)
2. turns the rule into a score with one reason code and one action label, then writes the ranked queue to `work/outputs/baseline_action_score.csv` (section 2)
3. reads my top ten with a skeptic's eye (sections 3 and 4)

**How to run it:** same as W03. In Colab save your Hugging Face READ token as a Secret named `HF_TOKEN`. Outside Colab the notebook reads the `HF_TOKEN` environment variable or asks for it with `getpass`. The token is never printed and never written into the notebook.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import getpass
from datetime import date
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, ndcg_score, roc_auc_score
from sklearn.model_selection import GroupKFold

# The repo is public, so the token comes from a Colab Secret or an environment variable.
# It is never printed. getpass is only the last fallback.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connected. Hugging Face token loaded (not printed).")

DuckDB connected. Hugging Face token loaded (not printed).


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule idea and the two signals it leans on

**Rule idea in plain words (before any check):** review first the pages that have real search demand and were already losing impressions in the second half of March.

That idea leans on two signals. Both are built from March only:

| | Signal | How it is built | Linked to a FlyRank flag? |
|---|---|---|---|
| A | **Volume**: search impressions per day in March | March impressions / 31 (W03's `imp_per_day_log` before the log) | **Yes**: volume is the signal behind the **quick-win** flag |
| B | **Late-March slide**: how much daily impressions fell from 1 to 15 March to 16 to 31 March, in % | from W03's `late_march_share`, written as a % drop so a person can read it | No, it is my own signal |

**Why not staleness**, the signal behind the refresh flags? The only edit dates in the release sit in `dim_content`, which is a July 2026 snapshot (W03 contract answer 5). A staleness check on 31 March would quietly use edits made later, so I leave it out. CTR vs position (behind the CTR-fix logic) would be a fair pick too, but my rule does not use it, so I test the two signals the rule really needs.

Each check gets a bucket table with n, a look at the 16 large clients (the W03 limitation said the label moves with the client) and a one word verdict: CONFIRMED, OPPOSITE, MIXED or FALSE.

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

# Same contract as W03: features from March, label from April
FEATURE_MONTH = "2026-03"          # feature window: 1 to 31 March 2026
LABEL_MONTH = "2026-04"            # label window: 1 to 30 April 2026
MARCH_DAYS, APRIL_DAYS = 31, 30
LATE_MARCH_START = "2026-03-16"    # second half of March starts here
EARLY_DAYS, LATE_DAYS = 15, 16     # 1 to 15 March and 16 to 31 March
GSC_TRACKED_BY = "2026-03-01"
MIN_MARCH_IMPRESSIONS = 100
DECLINE_CUTOFF_PCT = -20
LABEL = "declined_next_month"

# Rule thresholds (explained below the signal checks)
MIN_SLIDE_PCT = 20                 # same 20% cut as my label and the starter "down" rule
MIN_LATE_IMPRESSIONS_PER_DAY = 5   # still visible in late March
K_MAIN = 50                        # my main metric from W02: precision@50
SEED = 42
BIG_CLIENT_PAGES = 1000            # "large client" = 1,000+ pages in the frame (same cut as W03)
MIN_CELL_N = 30                    # no client comparison from a cell smaller than this

con.execute(f"CREATE OR REPLACE VIEW march AS SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet')")
con.execute(f"CREATE OR REPLACE VIEW april AS SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet')")
con.execute(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{REL}/dim_clients.parquet')")
print(f"Feature window: month={FEATURE_MONTH} | label window: month={LABEL_MONTH}")

Feature window: month=2026-03 | label window: month=2026-04


### Rebuild the W03 page frame

This is the W03 query with the same four eligibility rules. I only added two March-only columns to it (clicks in late March and the biggest single day in early March) plus a client rank for display. They are used in the review in section 3, never as rule inputs.

In [4]:
pages = con.sql(f"""
    WITH march_pages AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id)                           AS client_hash_id,  -- safe: W03 query 1 showed one client per page
            COUNT(*)                                            AS march_rows,
            SUM(gsc_impressions)::BIGINT                        AS imp_march,
            SUM(gsc_clicks)::BIGINT                             AS clicks_march,
            SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{LATE_MARCH_START}')::BIGINT AS imp_late_march,
            SUM(gsc_clicks)      FILTER (WHERE report_date >= DATE '{LATE_MARCH_START}')::BIGINT AS clicks_late_march,
            MAX(gsc_impressions) FILTER (WHERE report_date <  DATE '{LATE_MARCH_START}')::BIGINT AS early_peak_day,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_days,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_sum,
            SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_impressions
        FROM march
        GROUP BY content_hash_id
    ),
    april_pages AS (  -- label window only: no feature is built from this
        SELECT content_hash_id, COUNT(*) AS april_rows, SUM(gsc_impressions)::BIGINT AS imp_april
        FROM april
        GROUP BY content_hash_id
    )
    SELECT m.*, c.gsc_data_start, a.april_rows, a.imp_april
    FROM march_pages m
    LEFT JOIN dim_clients c USING (client_hash_id)
    LEFT JOIN april_pages a USING (content_hash_id)
""").df()

gsc_start = pd.to_datetime(pages["gsc_data_start"])
rules = [
    ("pages with rows in March", pd.Series(True, index=pages.index)),
    ("row on all 31 days of March", pages["march_rows"] == MARCH_DAYS),
    ("client GSC tracking started by 1 March", gsc_start <= pd.Timestamp(GSC_TRACKED_BY)),
    (f"at least {MIN_MARCH_IMPRESSIONS} GSC impressions in March", pages["imp_march"] >= MIN_MARCH_IMPRESSIONS),
    ("row on all 30 days of April (label measurable)", pages["april_rows"] == APRIL_DAYS),
]
keep = pd.Series(True, index=pages.index)
funnel = []
for rule_name, rule in rules:
    keep &= rule
    funnel.append((rule_name, int(keep.sum())))
eligible = pages[keep].sort_values("content_hash_id").reset_index(drop=True)
funnel = pd.DataFrame(funnel, columns=["rule (applied in order)", "pages left"])
funnel

,rule (applied in order),pages left
0,pages with rows in March,331437
1,row on all 31 days of March,275133
2,client GSC tracking started by 1 March,267986
3,at least 100 GSC impressions in March,94496
4,row on all 30 days of April (label measurable),94496


In [5]:
# March numbers per page (feature window only)
frame = pd.DataFrame({
    "content_hash_id": eligible["content_hash_id"],
    "client_hash_id": eligible["client_hash_id"],
    "imp_per_day": eligible["imp_march"] / MARCH_DAYS,
    "early_per_day": (eligible["imp_march"] - eligible["imp_late_march"]) / EARLY_DAYS,
    "late_per_day": eligible["imp_late_march"] / LATE_DAYS,
    "clicks_march": eligible["clicks_march"],
    "ctr_pct": 100 * eligible["clicks_march"] / eligible["imp_march"],
    "avg_position": eligible["position_sum"] / eligible["position_impressions"],
    "gsc_days": eligible["gsc_days"],
})
frame = frame.astype({c: "float64" for c in frame.columns if c not in ("content_hash_id", "client_hash_id")})

# Signal B as a % drop: positive = fell, negative = rose. No early impressions -> NaN (cannot fall)
frame["slide_pct"] = 100 * (1 - frame["late_per_day"] / frame["early_per_day"].where(frame["early_per_day"] > 0))
frame["lost_per_day"] = frame["early_per_day"] - frame["late_per_day"]

# Client rank: 1 = client with the most pages (same order as the W03 table). Hash ids stay out of every display
client_pages = frame.groupby("client_hash_id").size().reset_index(name="n").sort_values(["n", "client_hash_id"], ascending=[False, True])
client_rank = pd.Series(np.arange(1, len(client_pages) + 1), index=client_pages["client_hash_id"].to_numpy())
frame["client_rank"] = frame["client_hash_id"].map(client_rank)

# The label: April vs March per day (same as W03). The % change stays OUT of the frame, it is the answer
march_per_day = eligible["imp_march"] / MARCH_DAYS
april_per_day = eligible["imp_april"] / APRIL_DAYS
april_change_pct = (100 * (april_per_day - march_per_day) / march_per_day).astype("float64")
frame[LABEL] = (april_change_pct < DECLINE_CUTOFF_PCT).astype(int)
hindsight = pd.DataFrame({"content_hash_id": eligible["content_hash_id"],
                          "april_per_day": april_per_day, "april_vs_march_pct": april_change_pct})  # only for the review, after scoring

# Review-only context (March): clicks per half and the biggest single day in early March
early_impressions = eligible["imp_march"] - eligible["imp_late_march"]
context = pd.DataFrame({
    "content_hash_id": eligible["content_hash_id"],
    "clicks_early_per_day": (eligible["clicks_march"] - eligible["clicks_late_march"]) / EARLY_DAYS,
    "clicks_late_per_day": eligible["clicks_late_march"] / LATE_DAYS,
    "early_peak_share": eligible["early_peak_day"] / early_impressions.where(early_impressions > 0),
    "gsc_data_start": pd.to_datetime(eligible["gsc_data_start"]),
})

assert frame["content_hash_id"].is_unique, "grain broken: expected one row per page"
base_rate = frame[LABEL].mean()
print(f"Frame: {len(frame):,} pages from {frame['client_hash_id'].nunique()} clients | base rate of {LABEL}: {base_rate:.3f}")
print("Same as W03 (94,496 pages, 38 clients, base rate 0.516):",
      len(frame) == 94_496 and frame["client_hash_id"].nunique() == 38 and round(base_rate, 3) == 0.516)

Frame: 94,496 pages from 38 clients | base rate of declined_next_month: 0.516
Same as W03 (94,496 pages, 38 clients, base rate 0.516): True


### Signal check A: volume (the quick-win signal)

**Claim:** the more search impressions a page had in March, the more likely it is to lose 20%+ of its daily impressions in April. If this holds, putting big pages first (the quick-win habit) would also put the pages at risk first.

In [6]:
volume_bucket = pd.cut(frame["imp_per_day"], [0, 10, 30, 100, 300, 1000, np.inf], right=False,
                       labels=["under 10", "10 to 30", "30 to 100", "100 to 300", "300 to 1,000", "1,000+"])
table_a = (frame.groupby(volume_bucket, observed=True)[LABEL]
           .agg(n="size", declined="sum", decline_rate="mean"))
table_a.index.name = "March impressions per day"
table_a["vs base rate"] = (table_a["decline_rate"] - base_rate).map("{:+.3f}".format)
print(f"Base rate: {base_rate:.3f} | smallest bucket n = {table_a['n'].min():,}")
table_a.round(3)

Base rate: 0.516 | smallest bucket n = 902


,n,declined,decline_rate,vs base rate
March impressions per day,,,,
under 10,25054,13874,0.554,+0.038
10 to 30,24791,13565,0.547,+0.031
30 to 100,23537,12066,0.513,-0.003
100 to 300,14683,6509,0.443,-0.073
"300 to 1,000",5529,2379,0.430,-0.086
"1,000+",902,353,0.391,-0.124


In [7]:
def client_check(group_low, group_high, name_low, name_high):
    """Decline rate of two groups inside each large client. Cells under MIN_CELL_N pages are skipped."""
    rows = []
    for rank, g in frame.groupby("client_rank"):
        if len(g) < BIG_CLIENT_PAGES:
            continue
        low, high = g.loc[group_low[g.index], LABEL], g.loc[group_high[g.index], LABEL]
        if len(low) < MIN_CELL_N or len(high) < MIN_CELL_N:
            rows.append({"client rank": rank, "pages": len(g), f"n {name_low}": len(low), f"rate {name_low}": np.nan,
                         f"n {name_high}": len(high), f"rate {name_high}": np.nan})
            continue
        rows.append({"client rank": rank, "pages": len(g),
                     f"n {name_low}": len(low), f"rate {name_low}": low.mean(),
                     f"n {name_high}": len(high), f"rate {name_high}": high.mean()})
    return pd.DataFrame(rows).set_index("client rank")


by_client_a = client_check(frame["imp_per_day"] < 10, frame["imp_per_day"] >= 30, "under 10/day", "30+/day")
judged_a = by_client_a.dropna()
lower_for_big = int((judged_a["rate 30+/day"] < judged_a["rate under 10/day"]).sum())
print(f"Large clients judged: {len(judged_a)} of {len(by_client_a)} | big pages (30+/day) declined LESS often than small ones (under 10/day) in {lower_for_big} of {len(judged_a)}")
by_client_a.round(3)

Large clients judged: 16 of 16 | big pages (30+/day) declined LESS often than small ones (under 10/day) in 12 of 16


,pages,n under 10/day,rate under 10/day,n 30+/day,rate 30+/day
client rank,,,,,
1,21238,4891,0.390,10859,0.404
2,16337,4006,0.778,8276,0.687
3,11539,1272,0.435,7979,0.502
4,8501,1403,0.299,4447,0.283
5,7498,3904,0.356,1889,0.091
6,7368,1519,0.780,3664,0.660
7,3406,502,0.468,2200,0.411
8,2532,1092,0.627,760,0.379
9,2418,279,0.584,1540,0.325


**Verdict A (volume, the quick-win signal): OPPOSITE.**

The claim points the wrong way. The decline rate falls step by step from 55.4% for pages under 10 impressions a day to 39.1% for pages above 1,000 a day (every bucket has n ≥ 902). Inside the large clients it is the same story: big pages declined less often than small ones in 12 of the 16. Part of this is noise. At 5 impressions a day a 20% drop is one impression, so small pages cross the line much more easily.

**What it means in practice:** volume cannot be the thing that decides which pages are at risk. A quick-win style list that puts the biggest pages first would fill its top with steady pages (section 2 measures this). Volume can still help with something else, see "Putting them together" below.

### Signal check B: late-March slide

**Claim:** a page whose daily impressions already fell from the first half of March (1 to 15) to the second half (16 to 31) is more likely to lose 20%+ in April.

In [8]:
slide_bucket = pd.cut(frame["slide_pct"], [-np.inf, -20, 0, 10, 20, 35, 50, 75, 100.001], right=False,
                      labels=["rose 20%+", "rose 0 to 20%", "fell 0 to 10%", "fell 10 to 20%", "fell 20 to 35%",
                              "fell 35 to 50%", "fell 50 to 75%", "fell 75 to 100%"])
slide_bucket = slide_bucket.cat.add_categories("no impressions 1 to 15 March").fillna("no impressions 1 to 15 March")
table_b = frame.groupby(slide_bucket, observed=True)[LABEL].agg(n="size", declined="sum", decline_rate="mean")
table_b.index.name = "change from early to late March"
table_b["vs base rate"] = (table_b["decline_rate"] - base_rate).map("{:+.3f}".format)
print(f"Base rate: {base_rate:.3f} | smallest bucket n = {table_b['n'].min():,}")
table_b.round(3)

Base rate: 0.516 | smallest bucket n = 1,335


,n,declined,decline_rate,vs base rate
change from early to late March,,,,
rose 20%+,36151,13376,0.370,-0.146
rose 0 to 20%,13710,6451,0.471,-0.045
fell 0 to 10%,7618,3981,0.523,+0.007
fell 10 to 20%,7573,4369,0.577,+0.061
fell 20 to 35%,10289,6538,0.635,+0.120
fell 35 to 50%,7808,5534,0.709,+0.193
fell 50 to 75%,7356,6013,0.817,+0.302
fell 75 to 100%,2656,2299,0.866,+0.350
no impressions 1 to 15 March,1335,185,0.139,-0.377


In [9]:
slid = frame["slide_pct"] >= MIN_SLIDE_PCT          # NaN (no early impressions) counts as not slid
by_client_b = client_check(~slid, slid, "not slid", "slid 20%+")
judged_b = by_client_b.dropna()
higher_for_slid = int((judged_b["rate slid 20%+"] > judged_b["rate not slid"]).sum())
print(f"Large clients judged: {len(judged_b)} of {len(by_client_b)} | pages that slid 20%+ declined MORE often in {higher_for_slid} of {len(judged_b)}")
by_client_b.round(3)

Large clients judged: 16 of 16 | pages that slid 20%+ declined MORE often in 16 of 16


,pages,n not slid,rate not slid,n slid 20%+,rate slid 20%+
client rank,,,,,
1,21238,14537,0.323,6701,0.605
2,16337,9766,0.646,6571,0.854
3,11539,6975,0.356,4564,0.691
4,8501,7731,0.267,770,0.579
5,7498,6613,0.247,885,0.455
6,7368,5944,0.675,1424,0.897
7,3406,2466,0.324,940,0.698
8,2532,1796,0.427,736,0.764
9,2418,1683,0.291,735,0.707


**Verdict B (late-March slide): CONFIRMED.**

The decline rate climbs at every step, from 37.0% for pages that rose 20%+ to 86.6% for pages that fell 75% or more. Inside every one of the 16 large clients, pages that slid 20%+ declined more often than the rest. Pages with no impressions in early March (they only started showing up later) declined just 13.9% of the time, which also fits.

One honest note: part of this is simple carry-over. The label compares April with the whole March average. If a page just stays at its late-March level through April, it already counts as "declined" once its slide is above about 34% (worked out in the next cell). That is fine for a review queue, because the slide is known on 31 March. But it means this signal mostly finds pages that are already falling, not pages that are about to start.

In [10]:
# If April stays at the late-March level r (as a share of the early-March level), the label compares
#   r  with  the March average (15 * 1 + 16 * r) / 31.  Declined means r / that average < 0.8
r = 0.8 * EARLY_DAYS / (MARCH_DAYS - 0.8 * LATE_DAYS)
print(f"A page that just holds its late-March level is labelled 'declined' once its slide is above {100 * (1 - r):.1f}%")

A page that just holds its late-March level is labelled 'declined' once its slide is above 34.1%


### Putting them together

The two verdicts look like they clash: a slide predicts decline, but bigger pages decline less. The cross table below shows where the OPPOSITE result comes from.

In [11]:
volume_4 = pd.cut(frame["imp_per_day"], [0, 10, 30, 100, np.inf], right=False,
                  labels=["under 10", "10 to 30", "30 to 100", "100+"])
slid_label = np.where(slid, "slid 20%+", "did not slide 20%")
cross_rate = pd.crosstab(slid_label, volume_4, values=frame[LABEL], aggfunc="mean")
cross_n = pd.crosstab(slid_label, volume_4)
print("Decline rate (rows: late-March slide, columns: March impressions per day)")
display(cross_rate.round(3))
print("n per cell")
display(cross_n)

Decline rate (rows: late-March slide, columns: March impressions per day)


imp_per_day,under 10,10 to 30,30 to 100,100+
row_0,,,,
did not slide 20%,0.475,0.461,0.424,0.342
slid 20%+,0.725,0.731,0.723,0.720


n per cell


imp_per_day,under 10,10 to 30,30 to 100,100+
row_0,,,,
did not slide 20%,17175,16887,16564,15761
slid 20%+,7879,7904,6973,5353


Inside the pages that slid 20%+, volume does not change the risk: the decline rate stays around 72% in every volume bucket. Outside them, bigger pages are steadier (47.5% down to 34.2%). So the OPPOSITE verdict comes from pages that were not sliding. This tells me how to use each signal: **the slide decides who goes into the queue and volume only decides the order inside it.** There it costs nothing and puts the pages with more impressions at stake first.

### The rule (after the checks)

**In plain words:** a page goes into my review queue when its daily search impressions fell at least 20% from the first half of March to the second half and it still had at least 5 impressions a day in the second half. Inside the queue, the page that lost the most impressions per day comes first. Every other page is left to monitor.

| Part | Value |
|---|---|
| Score `action_score` | impressions lost per day = early-March per day minus late-March per day, only when the rule fires. 0 otherwise |
| Reason code (one per page) | `sliding_with_demand` when the rule fires, `no_rule_fired` otherwise |
| Action label | `review_for_refresh` when the rule fires, `monitor` otherwise |

**Why these thresholds:**
- **20% slide:** the same 20% cut as my label and the starter `down` rule. I did not tune it.
- **5 impressions a day in late March:** close to the starter's "visible" bar (500 impressions in 90 days is about 5.6 a day). It also keeps out pages that have already vanished. A page that dropped to almost zero in mid March needs a technical check (removed, redirected or dropped from the index), not a refresh, so `review_for_refresh` would be the wrong action for it.

**What the checks changed:** my first idea was quick-win style, with bigger pages first. Verdict A says volume points the wrong way when it decides who is at risk, so in the final rule it only orders pages after the slide has picked them.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
RULE_INPUTS = ["early_per_day", "late_per_day"]   # both come from the March partition only


def sliding_with_demand(pages: pd.DataFrame) -> pd.DataFrame:
    """The baseline rule. Reads only March columns. Returns a score, one reason code and one action per page."""
    early, late = pages["early_per_day"], pages["late_per_day"]
    slide_pct = 100 * (1 - late / early.where(early > 0))                 # NaN when there were no early impressions
    fires = slide_pct.ge(MIN_SLIDE_PCT) & late.ge(MIN_LATE_IMPRESSIONS_PER_DAY)
    return pd.DataFrame({
        "action_score": np.where(fires, early - late, 0.0),
        "reason_code": np.where(fires, "sliding_with_demand", "no_rule_fired"),
        "action": np.where(fires, "review_for_refresh", "monitor"),
    }, index=pages.index)


# The rule only ever sees a March-only copy: no April column and no label
march_only = frame[["content_hash_id"] + RULE_INPUTS].copy()
assert LABEL not in march_only.columns and not any("april" in c for c in march_only.columns)
scored = sliding_with_demand(march_only)

QUEUE_COLUMNS = ["queue_rank", "content_hash_id", "client_hash_id", "action_score", "reason_code", "action",
                 "imp_per_day", "early_per_day", "late_per_day", "slide_pct", "lost_per_day",
                 "clicks_march", "ctr_pct", "avg_position", "gsc_days"]
queue = (frame.drop(columns=[LABEL]).join(scored)
         .sort_values(["action_score", "content_hash_id"], ascending=[False, True])   # ties (score 0) in a fixed order
         .reset_index(drop=True))
queue["queue_rank"] = np.arange(1, len(queue) + 1)
queue = queue[QUEUE_COLUMNS]

# Works from work/notebooks/ (local clone) and from Colab (/content)
OUT_DIR = Path("../outputs") if Path.cwd().name == "notebooks" else Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
queue_path = OUT_DIR / "baseline_action_score.csv"
queue.round(3).to_csv(queue_path, index=False)

fired = queue["reason_code"].eq("sliding_with_demand")
print(f"Wrote {len(queue):,} ranked pages to {queue_path}")
print(f"Rule fired on {fired.sum():,} pages ({fired.mean():.1%}). The other {(~fired).sum():,} get score 0, no_rule_fired, monitor")
print(queue["action"].value_counts().to_string())
queue.drop(columns=["client_hash_id"]).head(5).round(2)

Wrote 94,496 ranked pages to ../outputs/baseline_action_score.csv
Rule fired on 21,855 pages (23.1%). The other 72,641 get score 0, no_rule_fired, monitor
action
monitor               72641
review_for_refresh    21855


,queue_rank,content_hash_id,action_score,reason_code,action,imp_per_day,early_per_day,late_per_day,slide_pct,lost_per_day,clicks_march,ctr_pct,avg_position,gsc_days
0,1,content_e8a52cf3d5988c07,3184.99,sliding_with_demand,review_for_refresh,7901.00,9544.87,6359.88,33.37,3184.99,669.0,0.27,15.17,31.0
1,2,content_65c75874a23fca87,3133.56,sliding_with_demand,review_for_refresh,2094.68,3712.00,578.44,84.42,3133.56,18.0,0.03,7.61,31.0
2,3,content_7c6373141eae744a,2932.35,sliding_with_demand,review_for_refresh,4277.19,5790.67,2858.31,50.64,2932.35,83.0,0.06,5.95,31.0
3,4,content_62673eea26c31c17,2771.52,sliding_with_demand,review_for_refresh,1861.94,3292.40,520.88,84.18,2771.52,43.0,0.07,6.02,31.0
4,5,content_945d6ff91386c817,2727.35,sliding_with_demand,review_for_refresh,1879.94,3287.60,560.25,82.96,2727.35,5.0,0.01,8.33,31.0


### How good is the queue? Precision@K against the floors

Same metric as W02: **precision@50** is the main number, with @20, @100, average precision, NDCG@50 and ROC AUC as support. Two floors sit under the rule:
- **random order**: `DummyClassifier(strategy="prior")` gives every page the same score, so its AP, ROC AUC and NDCG are the "no skill" values. For precision@K I shuffle the pages 1,000 times (seed 42) and take the average.
- **volume only**: sort every page by March impressions per day, the quick-win way. This is what verdict A warned about.

The rule never sees the label. I join it only now, by `content_hash_id`, to score the queue.

In [13]:
label_by_page = frame.set_index("content_hash_id")[LABEL]


def precision_at_k(ranked_labels, k):
    """Share of declined pages among the first k pages of a ranked list."""
    return float(np.asarray(ranked_labels)[:k].mean())


def queue_metrics(ranked_labels, scores):
    y, s = np.asarray(ranked_labels), np.asarray(scores, dtype=float)
    return {"P@20": precision_at_k(y, 20), "P@50": precision_at_k(y, 50), "P@100": precision_at_k(y, 100),
            "avg precision": average_precision_score(y, s), "NDCG@50": ndcg_score(y[None, :], s[None, :], k=50),
            "ROC AUC": roc_auc_score(y, s)}


# 1) my rule, in queue order
rule_labels = queue["content_hash_id"].map(label_by_page).to_numpy()
rule = queue_metrics(rule_labels, queue["action_score"])

# 2) volume only (quick-win style)
by_volume = frame.sort_values(["imp_per_day", "content_hash_id"], ascending=[False, True])
volume = queue_metrics(by_volume[LABEL], by_volume["imp_per_day"])

# 3) random order: DummyClassifier for the score-based metrics, 1,000 shuffles for precision@K
y_all = frame[LABEL].to_numpy()
dummy_scores = DummyClassifier(strategy="prior").fit(np.zeros((len(y_all), 1)), y_all).predict_proba(np.zeros((len(y_all), 1)))[:, 1]
random_order = queue_metrics(y_all, dummy_scores)                     # P@K here is overwritten just below
rng = np.random.default_rng(SEED)
shuffled = np.array([[precision_at_k(y_all[rng.permutation(len(y_all))], k) for k in (20, 50, 100)] for _ in range(1000)])
random_order.update({"P@20": shuffled[:, 0].mean(), "P@50": shuffled[:, 1].mean(), "P@100": shuffled[:, 2].mean()})
p50_low, p50_high = np.percentile(shuffled[:, 1], [5, 95])

results = pd.DataFrame({"random order (DummyClassifier floor)": random_order,
                        "volume only (quick-win style)": volume,
                        "my rule: sliding_with_demand": rule}).T
print(f"Pages: {len(frame):,} | base rate: {base_rate:.3f} | random P@50, 5th to 95th percentile of 1,000 shuffles: {p50_low:.2f} to {p50_high:.2f}")
results.round(3)

Pages: 94,496 | base rate: 0.516 | random P@50, 5th to 95th percentile of 1,000 shuffles: 0.40 to 0.64


,P@20,P@50,P@100,avg precision,NDCG@50,ROC AUC
random order (DummyClassifier floor),0.514,0.515,0.514,0.516,0.516,0.500
volume only (quick-win style),0.300,0.280,0.330,0.476,0.298,0.452
my rule: sliding_with_demand,0.800,0.840,0.870,0.593,0.842,0.596


**Same 5 client folds as W03.** My Week-5 model will be tested with `GroupKFold(n_splits=5)` on `client_hash_id`, like the W03 quick score. A rule has nothing to train, so its fold number is just the rule applied to each test fold. These are the per-fold numbers the model has to beat on the same pages.

In [14]:
fold_rows = []
for fold, (_, test_idx) in enumerate(GroupKFold(n_splits=5).split(frame, frame[LABEL], groups=frame["client_hash_id"])):
    test_pages = set(frame.loc[test_idx, "content_hash_id"])
    fold_queue = queue[queue["content_hash_id"].isin(test_pages)]          # already in rule order
    fold_labels = fold_queue["content_hash_id"].map(label_by_page)
    fold_rows.append({"fold": fold, "clients": frame.loc[test_idx, "client_hash_id"].nunique(), "pages": len(test_idx),
                      "base rate": fold_labels.mean(), "pages fired": int(fold_queue["reason_code"].eq("sliding_with_demand").sum()),
                      "rule P@50": precision_at_k(fold_labels, K_MAIN)})
folds = pd.DataFrame(fold_rows).set_index("fold")
print(f"Rule P@50 across the 5 client folds: mean {folds['rule P@50'].mean():.3f}, from {folds['rule P@50'].min():.2f} to {folds['rule P@50'].max():.2f}")
folds.round(3)

Rule P@50 across the 5 client folds: mean 0.784, from 0.62 to 0.92


,clients,pages,base rate,pages fired,rule P@50
fold,,,,,
0,1,21238,0.412,5614,0.80
1,7,18314,0.725,5292,0.92
2,9,18314,0.486,5645,0.88
3,10,18317,0.428,2939,0.70
4,11,18313,0.545,2365,0.62


In [15]:
# The run's receipt: small, no page ids, safe to commit (CSV files stay out of git by design)
metrics = {
    "notebook": "work/notebooks/w04_baseline_score.ipynb",
    "run_date": str(date.today()),
    "data": {"release": "flyrank_pseudonymized_warehouse_release_v20260703",
             "feature_window": "2026-03-01 to 2026-03-31", "label_window": "2026-04-01 to 2026-04-30",
             "label": f"{LABEL}: April impressions per day more than 20% below March", "pages": int(len(frame)),
             "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(float(base_rate), 4)},
    "rule": {"name": "sliding_with_demand", "inputs": RULE_INPUTS,
             "fires_when": f"slide from 1-15 March to 16-31 March >= {MIN_SLIDE_PCT}% and late-March impressions per day >= {MIN_LATE_IMPRESSIONS_PER_DAY}",
             "score": "early-March impressions per day minus late-March impressions per day (0 when the rule does not fire)",
             "reason_codes": ["sliding_with_demand", "no_rule_fired"], "actions": ["review_for_refresh", "monitor"],
             "pages_fired": int(fired.sum())},
    "signal_verdicts": {"A_volume_quick_win": "OPPOSITE", "B_late_march_slide": "CONFIRMED"},
    "all_pages": {name: {m: round(float(v), 4) for m, v in row.items()} for name, row in results.iterrows()},
    "random_p50_5th_95th_percentile": [round(float(p50_low), 3), round(float(p50_high), 3)],
    "client_folds": {"split": "GroupKFold(n_splits=5) on client_hash_id",
                     "rule_p50_per_fold": [round(float(v), 3) for v in folds["rule P@50"]],
                     "rule_p50_mean": round(float(folds["rule P@50"].mean()), 4)},
}
metrics_path = OUT_DIR / "w04_baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Wrote {metrics_path}")

Wrote ../outputs/w04_baseline_metrics.json


**What the numbers say.** Random order gets P@50 = 0.515 (the base rate) and 90% of shuffles land between 0.40 and 0.64. Sorting by volume does worse than random (P@50 = 0.28), which is verdict A showing up as a number. My rule gets **P@50 = 0.84**: 42 of its top 50 pages lost 20%+ in April. On the five client folds it averages 0.784, but the folds go from 0.62 to 0.92, so one P@50 number moves a lot with which clients are tested.

The rule is built for the top of the queue. It gives 0 to the 72,641 pages where it does not fire and has no opinion about their order, which is why its ROC AUC (0.596) is low and below the W03 tree's 0.692. **The numbers my Week-5 model must beat are P@50 = 0.84 on all pages and 0.784 on the client folds.** Section 4 shows these numbers flatter the rule, so the top ten matter more than the score.

## 3. Top-10 review

*For each of the top 10: the action, why it's there and what would make it wrong.*

To read the top with a skeptic's eye I add one piece of context the rule never uses: **each page's February 2026 level** (impressions per day). February is before my decision moment (31 March), so a reviewer would have it. It answers the question the rule can't ask: is late March below this page's normal level, or did early March just have a surge? I read February only for the pages the rule picked and only after the queue is written. Section 4 uses the same numbers.

The April numbers in the last three columns are hindsight. The rule never saw them. Under the table I also print the daily March impressions of the top ten, so the shape of each "slide" is visible.

In [16]:
REVIEW_MONTH = "2026-02"            # context for the human review only, never a rule input
TOP_REVIEW = 50
picked = queue[queue["reason_code"].eq("sliding_with_demand")]

con.register("picked_pages", picked[["content_hash_id"]])
feb = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS feb_rows, SUM(gsc_impressions)::BIGINT AS imp_feb
    FROM read_parquet('{REL}/fact_content_daily_performance/month={REVIEW_MONTH}/*.parquet')
    JOIN picked_pages USING (content_hash_id)
    GROUP BY content_hash_id
""").df()

picked = (picked.merge(feb, on="content_hash_id", how="left").merge(context, on="content_hash_id")
          .merge(hindsight, on="content_hash_id").sort_values("queue_rank").reset_index(drop=True))
picked["client_rank"] = picked["client_hash_id"].map(client_rank)
# February level only when the page has all 28 days, its client was tracked from 1 February and it had impressions
full_feb = picked["feb_rows"].eq(28) & picked["gsc_data_start"].le(pd.Timestamp("2026-02-01")) & picked["imp_feb"].gt(0)
picked["feb_per_day"] = (picked["imp_feb"] / 28).where(full_feb)
picked["late_vs_feb"] = picked["late_per_day"] / picked["feb_per_day"]
picked[LABEL] = picked["content_hash_id"].map(label_by_page)
top = picked.head(TOP_REVIEW)
print(f"February level available for {full_feb.sum():,} of {len(picked):,} picked pages ({top['feb_per_day'].notna().sum()} of the top {TOP_REVIEW})")

review = top.head(10)[["queue_rank", "client_rank", "action", "feb_per_day", "early_per_day", "late_per_day", "slide_pct",
                       "lost_per_day", "late_vs_feb", "early_peak_share", "clicks_early_per_day", "clicks_late_per_day",
                       "ctr_pct", "avg_position", "april_per_day", "april_vs_march_pct", LABEL]]
review = review.rename(columns={"april_per_day": "april_per_day (hindsight)", "april_vs_march_pct": "april_vs_march_% (hindsight)",
                                LABEL: f"{LABEL} (hindsight)"}).set_index("queue_rank")
review.round({"feb_per_day": 0, "early_per_day": 0, "late_per_day": 0, "slide_pct": 0, "lost_per_day": 0, "late_vs_feb": 2,
              "early_peak_share": 2, "clicks_early_per_day": 1, "clicks_late_per_day": 1, "ctr_pct": 2, "avg_position": 1,
              "april_per_day (hindsight)": 0, "april_vs_march_% (hindsight)": 1})

February level available for 17,114 of 21,855 picked pages (43 of the top 50)


,client_rank,action,feb_per_day,early_per_day,late_per_day,slide_pct,lost_per_day,late_vs_feb,early_peak_share,clicks_early_per_day,clicks_late_per_day,ctr_pct,avg_position,april_per_day (hindsight),april_vs_march_% (hindsight),declined_next_month (hindsight)
queue_rank,,,,,,,,,,,,,,,,
1,3,review_for_refresh,5790.0,9545.0,6360.0,33.0,3185.0,1.10,0.11,23.5,19.8,0.27,15.2,5790.0,-26.7,1
2,3,review_for_refresh,127.0,3712.0,578.0,84.0,3134.0,4.56,0.2,1.0,0.2,0.03,7.6,775.0,-63.0,1
3,2,review_for_refresh,4157.0,5791.0,2858.0,51.0,2932.0,0.69,0.15,3.4,2.0,0.06,5.9,1645.0,-61.5,1
4,18,review_for_refresh,1611.0,3292.0,521.0,84.0,2772.0,0.32,0.11,2.3,0.5,0.07,6.0,1990.0,6.9,0
5,2,review_for_refresh,124.0,3288.0,560.0,83.0,2727.0,4.52,0.76,0.1,0.2,0.01,8.3,102.0,-94.6,1
6,2,review_for_refresh,365.0,3492.0,810.0,77.0,2682.0,2.22,0.47,1.2,1.0,0.05,4.6,122.0,-94.2,1
7,1,review_for_refresh,130.0,2773.0,526.0,81.0,2247.0,4.06,0.14,1.7,0.7,0.07,9.0,1501.0,-6.9,0
8,3,review_for_refresh,1364.0,3530.0,1321.0,63.0,2209.0,0.97,0.11,1.4,1.3,0.06,30.2,1344.0,-43.8,1
9,3,review_for_refresh,2436.0,5603.0,3507.0,37.0,2096.0,1.44,0.11,8.0,4.8,0.14,23.6,2848.0,-37.0,1


In [17]:
# Daily March impressions for the top ten: context for the lines below (March only, read after scoring)
con.register("top10_pages", top.head(10)[["content_hash_id", "queue_rank"]])
daily = con.sql("""
    SELECT t.queue_rank, EXTRACT(day FROM m.report_date)::INT AS march_day, m.gsc_impressions
    FROM march m JOIN top10_pages t USING (content_hash_id)
""").df()
daily_wide = daily.pivot(index="queue_rank", columns="march_day", values="gsc_impressions")
daily_wide.index.name = "rank / March day"
with pd.option_context("display.max_columns", 40):
    display(daily_wide)

march_day,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31
rank / March day,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,5696,9993,11019,11347,10134,7936,6066,8471,15394,13060,13910,14138,7983,3646,4380,7443,7784,6836,6554,4378,2857,4077,7971,8511,8551,7590,5980,3235,4204,7779,8008
2,192,706,378,591,4251,5822,2800,3588,11325,6377,3503,5529,9885,413,320,1388,409,317,274,199,232,231,192,242,408,930,492,670,960,1233,1078
3,3929,12648,3451,9420,4338,2069,5867,7858,9987,8545,8773,4236,2064,975,2700,2051,2224,2731,2529,2063,1812,2807,3068,2857,3357,3527,2584,2771,3720,4247,3385
4,3282,5128,3427,2850,2686,3249,3920,3653,5364,5072,3254,3581,3015,646,259,276,383,436,253,340,356,339,539,575,602,856,759,651,432,625,912
5,282,912,1910,37368,1750,92,80,162,96,79,140,125,218,942,5158,5102,86,145,100,674,395,1755,79,78,162,42,67,72,61,49,97
6,235,403,541,7810,24456,296,295,260,209,208,399,389,253,170,16454,5433,241,247,252,754,770,3166,337,332,261,285,193,181,184,197,119
7,638,668,771,761,3311,5753,4959,5271,4585,4351,2496,3545,2620,828,1033,1161,778,265,231,236,256,247,250,216,728,1163,967,589,444,483,403
8,2350,2778,2856,3638,4635,3595,2782,3756,4722,5430,5236,5595,2790,1033,1752,2061,1747,1282,1306,1019,803,1133,1630,1517,1457,1315,1169,835,1220,1345,1299
9,2265,4293,5278,6887,6777,5478,2990,4638,8267,9300,8799,9274,5146,2064,2585,4708,4960,4237,3768,2763,1586,1935,3906,4167,4315,4254,3609,1693,2105,3949,4160


**My top ten, one line each.** All ten get the action `review_for_refresh` and the reason code `sliding_with_demand`. Impressions are per day.

| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | review_for_refresh | Biggest loss in the queue: 9,545 → 6,360 a day (−33%) at position 15 and by far the most clicks in the top ten (about 20 a day) | Late March is still about 10% above its February level (5,790), so this may be a surge ending. Its client also slid 14.5% site wide |
| 2 | review_for_refresh | Lost 3,134 a day (3,712 → 578, −84%) | February was only 127 a day, so early March was a short surge. Late March is still 4.6× February and it gets under 1 click a day |
| 3 | review_for_refresh | Lost 2,932 a day (5,791 → 2,858, −51%) and late March is about 30% below February, so this is a real drop below its normal level | Clicks only went from 3.4 to 2.0 a day (CTR 0.06% at about position 6), so these impressions may come from searches that hardly ever click |
| 4 | review_for_refresh | Lost 2,772 a day (3,292 → 521, −84%) and late March is 68% below February, the clearest real drop in the top ten | It fell from about 3,000 to about 650 in one day (14 March). A cliff like that looks like an indexing or tracking hiccup that can fix itself, not slow content decay |
| 5 | review_for_refresh | Lost 2,727 a day on paper (3,288 → 560, −83%) | One day holds 76% of its early-March impressions. February was 124 a day, so late March is still 4.5× normal and it gets almost no clicks (CTR 0.01%) |
| 6 | review_for_refresh | Lost 2,682 a day (3,492 → 810, −77%) | Early March is mostly spike days (one day alone holds 47%). Late March is still 2.2× its February level (365) and clicks stayed around 1 a day |
| 7 | review_for_refresh | Lost 2,247 a day (2,773 → 526, −81%) | February was 130 a day, so this is a surge that ended. Late March is still 4.1× February and it gets under 2 clicks a day |
| 8 | review_for_refresh | Lost 2,209 a day (3,530 → 1,321, −63%) | Late March (1,321) is back at its February level (1,364), so the "slide" is just a surge ending. At position 30 clicks stayed flat (1.4 → 1.3 a day) |
| 9 | review_for_refresh | Lost 2,096 a day (5,603 → 3,507, −37%) and clicks fell from 8.0 to 4.8 a day, one of the few real click losses | Late March is still 44% above February (2,436), so the lost clicks may just be surge clicks going away |
| 10 | review_for_refresh | Lost 2,035 a day (7,327 → 5,292, −28%) | Clicks did not drop at all (7.7 → 7.9 a day) and late March is 47% above February. At position 33 most of these impressions never turn into visits |

**Hindsight (April, not used by the rule):** 7 of the 10 declined. Rank 4 came back (+6.9%), rank 7 held (−6.9%) and rank 10 missed the cutoff by 0.4 points (−19.6%). Two of the "hits" are not real drops either: rank 1 (−26.7%) and rank 5 (−94.6%) only fell back to about their February level (5,790 and 102 a day in April vs 5,790 and 124 in February).

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# Check 1: how much of the queue is a surge ending? ("surge ended" = late March still at or above the February level)
judged = picked[picked["feb_per_day"].notna()].copy()
judged["surge_ended"] = judged["late_per_day"] >= judged["feb_per_day"]
depth_rows = []
for k in [50, 100, 500, 1000, 5000, len(picked)]:
    part = judged[judged["queue_rank"] <= k]
    surge, real = part[part["surge_ended"]], part[~part["surge_ended"]]
    depth_rows.append({"part of the queue": f"top {k:,}" if k < len(picked) else f"all {k:,} picks",
                       "pages with a Feb level": len(part), "share surge ended": part["surge_ended"].mean(),
                       "n surge ended": len(surge), "declined: surge ended": surge[LABEL].mean(),
                       "n real drop": len(real), "declined: real drop": real[LABEL].mean()})
depth = pd.DataFrame(depth_rows).set_index("part of the queue")
print("Decline rates are hindsight (April). The rule never saw them.")
display(depth.round(3))

# Check 2: other weak spots in the top 50
top_client = top["client_rank"].value_counts()
lead_rank, lead_count = int(top_client.index[0]), int(top_client.iloc[0])
lead = frame[frame["client_rank"] == lead_rank]
lead_slide = 100 * (1 - lead["late_per_day"].sum() / lead["early_per_day"].sum())
all_slide = 100 * (1 - frame["late_per_day"].sum() / frame["early_per_day"].sum())
weak = pd.DataFrame([
    ("one day holds 30%+ of the early-March impressions", f"{(top['early_peak_share'] >= 0.30).sum()} of {len(top)}"),
    ("fewer than 31 clicks in all of March (under 1 a day)", f"{(top['clicks_march'] < 31).sum()} of {len(top)}"),
    (f"from one client (client rank {lead_rank})", f"{lead_count} of {len(top)}"),
    (f"client rank {lead_rank}: site-wide slide, early to late March", f"{lead_slide:.1f}% (all pages together: {all_slide:.1f}%)"),
    ("hindsight: did not decline in April", f"{(top[LABEL] == 0).sum()} of {len(top)}"),
], columns=["top-50 check", "result"]).set_index("top-50 check")
with pd.option_context("display.max_colwidth", 80):
    display(weak)

Decline rates are hindsight (April). The rule never saw them.


,pages with a Feb level,share surge ended,n surge ended,declined: surge ended,n real drop,declined: real drop
part of the queue,,,,,,
top 50,43,0.744,32,0.906,11,0.727
top 100,90,0.678,61,0.902,29,0.828
top 500,431,0.555,239,0.837,192,0.823
"top 1,000",862,0.516,445,0.809,417,0.835
"top 5,000",4124,0.369,1522,0.748,2602,0.798
"all 21,855 picks",17114,0.311,5320,0.721,11794,0.716


,result
top-50 check,
one day holds 30%+ of the early-March impressions,4 of 50
fewer than 31 clicks in all of March (under 1 a day),23 of 50
from one client (client rank 3),35 of 50
"client rank 3: site-wide slide, early to late March",14.5% (all pages together: -8.6%)
hindsight: did not decline in April,8 of 50


**Weak picks I found:**

1. **A surge, not decay.** In the top 50, 32 of the 43 pages with a February level are still at or above that level in late March. They had a short surge in early March and then went back toward normal. A refresh can't win back a surge. Ranks 2, 5, 6 and 7 are the clearest cases (late March is still 2.2 to 4.6 times their February level). The share drops as I go down the queue (74% of the top 50, 55% of the top 500, 31% of all picks), so ordering by impressions lost per day is exactly what pulls surges to the top. The rule only compares the two halves of March, so it reads "surge ended" as "sliding".
2. **Almost no clicks at stake.** 23 of the top 50 got fewer than 31 clicks in all of March. The rule counts impressions, but a reviewer cares about visits: losing 3,000 impressions a day at CTR 0.03% costs about one click a day.
3. **One client fills the list.** 35 of the top 50 come from client rank 3. Its whole site lost 14.5% of its daily impressions from early to late March, while all pages together grew 8.6%. That looks like one site-level change, not 35 separate page problems.
4. **Hindsight misses.** 8 of the top 50 did not decline in April. Rank 4 is the pick I would have trusted most (a real drop below February) and it came back.

**The bigger problem sits in the label, not only in the rule.** Over all picks with a February level, the label says "declined" for 72.1% of the surge-ended pages and 71.6% of the real drops (n = 5,320 and 11,794). So the label cannot tell a surge ending from a real fall below normal: it compares April with the March average and a surge pushes that average up. In the top 50 the surge pages even score higher (90.6% vs 72.7%, but there are only 11 real drops, below my 30 floor, so that part is just a hint). This means **P@50 = 0.84 makes the top of my queue look more useful than it is**: most of those hits are surges ending, which a refresh can't fix. A Week-5 model trained on this label gets rewarded for the same thing.

I am keeping the rule and the label frozen as they are, so the Week-5 model faces exactly the same test. Two notes for later. First, give the model a pre-March level (like February) so it can tell a surge from a real drop. Second, in W06 re-check the result with a label that compares against a longer baseline than one month.

In [19]:
# Leakage check: every line must pass
checks = {}

# 1) The rule reads only March columns and the copy it saw had no label and no April column
checks["rule inputs are March-only"] = set(RULE_INPUTS) <= {"early_per_day", "late_per_day"}
checks["rule input copy has no label or April column"] = LABEL not in march_only.columns and not any("april" in c for c in march_only.columns)

# 2) The written queue holds no label, no April number and no label-derived column
written = pd.read_csv(queue_path, nrows=5).columns
checks["CSV has no label / April / change column"] = not any(c == LABEL or "april" in c or "change" in c for c in written)

# 3) The one eligibility rule that touches April (label measurable) removed no page, so it does not shape the queue
checks["April eligibility rule removed 0 pages"] = int(funnel["pages left"].iloc[-2]) == int(funnel["pages left"].iloc[-1])

# 4) Same score with the label shuffled away from the pages: the rule does not depend on it at all
shuffled_frame = march_only.assign(**{LABEL: np.random.default_rng(SEED).permutation(frame[LABEL].to_numpy())})
checks["score unchanged when the label is shuffled"] = sliding_with_demand(shuffled_frame)["action_score"].equals(scored["action_score"])

# 5) No product flags exist in the fact table to leak (health_score, quick-win, CTR-fix, priority, action type...)
fact_columns = con.sql("DESCRIBE march").df()["column_name"].str.lower()
flag_words = ["health", "priority", "quick", "flag", "action", "needs", "tier", "score"]
checks["no product flag columns in the fact table"] = not fact_columns.str.contains("|".join(flag_words)).any()

# 6) The score is not secretly the answer: compare with W03's leak, which hit ROC AUC 1.000
checks["rule ROC AUC far from 1.0 (no copied answer)"] = rule["ROC AUC"] < 0.9

for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
assert all(checks.values()), "a leakage check failed"
print(f"\nAll {len(checks)} checks passed. February was read only in section 3, after the queue was written.")

PASS  rule inputs are March-only
PASS  rule input copy has no label or April column
PASS  CSV has no label / April / change column
PASS  April eligibility rule removed 0 pages
PASS  score unchanged when the label is shuffled
PASS  no product flag columns in the fact table
PASS  rule ROC AUC far from 1.0 (no copied answer)

All 7 checks passed. February was read only in section 3, after the queue was written.


**Leakage summary.** The rule uses two March numbers per page and nothing else. It never saw April or the label (the shuffle test gives the same score). The only eligibility rule that touches April removed no pages. The release ships no product flags, so `health_score`, quick-win or CTR-fix outputs could not slip in. Its ROC AUC is 0.596, nowhere near the 1.000 of the W03 leak. February appears only in the human review after the queue is written. It is also in the past, so it can't leak the future. The real weakness is not leakage, it is the surge problem above.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Checked against "what done looks like":**

- [x] Two signal verdicts with visible bucket tables and n: A volume (flag-linked, quick-win) = **OPPOSITE**, B late-March slide = **CONFIRMED**
- [x] One rule with a score (`action_score`), one reason code (`sliding_with_demand`) and an action label (`review_for_refresh` / `monitor`)
- [x] The ranked queue is written from the notebook to `work/outputs/baseline_action_score.csv` (gitignored, regenerated on every run) with the receipt `work/outputs/w04_baseline_metrics.json`
- [x] Ten reviewed rows, each with "what would make it wrong"
- [x] No future-window or label-derived inputs (seven checks pass in section 4)
- [x] Lane confirmed: Lane 2, Refresh / Content Opportunity Scoring